In [4]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef,accuracy_score, roc_auc_score,precision_score,f1_score,confusion_matrix,roc_curve
from sklearn.model_selection import StratifiedKFold

from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold

from collections import defaultdict
import json
import random
import os
from typing import List, Tuple, Dict, Optional
import math
import re
import copy
import optuna
import gc

from warnings import filterwarnings
# Silence some expected warnings
filterwarnings("ignore")

In [5]:
# ============ 1. Set the global random seed  ============
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [6]:
# ============ 2. Tokenizer ============
smiles_token_pattern = (
    r"(\[[^\]]+]"           # bracket atoms/groups
    r"|Br?|Cl?"             # halogens
    r"|N|O|S|P|F|I|b|c|n|o|s|p"
    r"|\(|\)|\."            # punctuation
    r"|=|#|-|\+|\\|/"
    r"|:|~|@@|@"            # chirality
    r"|\?|>>?"
    r"|\*|\$"
    r"|\%[0-9]{2}"          # %10 etc.
    r"|[0-9])"
)
tokenizer_re = re.compile(smiles_token_pattern)

def tokenize(smiles: str):
    if smiles is None:
        return []
    s = str(smiles).strip()
    if not s:
        return []
    return tokenizer_re.findall(s)

In [7]:
# ============ 3. Vectorization ============
def smiles_to_ids(smiles, vocab, unk_token="<UNK>", bos_token=None, eos_token=None):
    toks = tokenize(smiles)
    ids = [vocab.get(t, vocab[unk_token]) for t in toks]
    if bos_token is not None:
        ids = [vocab[bos_token]] + ids
    if eos_token is not None:
        ids = ids + [vocab[eos_token]]
    return torch.tensor(ids, dtype=torch.long)

def collate_smiles_to_batch(smiles_batch, vocab, pad_token="<PAD>", bos_token=None, eos_token=None):
    seqs = [smiles_to_ids(s, vocab, bos_token=bos_token, eos_token=eos_token) for s in smiles_batch]
    padded = pad_sequence(seqs, batch_first=True, padding_value=vocab[pad_token])
    pad_mask = (padded == vocab[pad_token]).to(torch.bool)
    return padded, pad_mask

In [8]:
# ============ 4. Create Dataset ============
class SMILESDataset(Dataset):
    def __init__(self, df):
        self.smiles = ["" if pd.isna(x) else str(x) for x in df["Smiles"].tolist()]
        self.labels = df["Label"].astype(float).tolist()

    def __getitem__(self, idx):
        return self.smiles[idx], torch.tensor(self.labels[idx], dtype=torch.float32)

    def __len__(self):
        return len(self.smiles)

def make_collate_fn(vocab, use_bos_eos=True):
    bos = "<BOS>" if use_bos_eos else None
    eos = "<EOS>" if use_bos_eos else None
    def collate_fn(batch):
        smiles_batch, labels = zip(*batch)
        padded, pad_mask = collate_smiles_to_batch(list(smiles_batch), vocab,
                                                  pad_token="<PAD>", bos_token=bos, eos_token=eos)
        labels = torch.stack(labels)
        return padded, pad_mask, labels
    return collate_fn

In [9]:
# ============ 5. Positional Encoding ============
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000, learn_scale=True):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(1))
        self.alpha = nn.Parameter(torch.tensor(1.0)) if learn_scale else None

    def forward(self, x):
        pe = self.pe[:x.size(0)].to(dtype=x.dtype)
        return x + (self.alpha * pe if self.alpha is not None else pe)

In [10]:
# ============ 6. Transformer Classifier ============
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, pos_encoder, nhead=4, d_model=128, dim_feedforward=512,
                 dropout=0.3, num_layers=2, use_cls_token=True, pad_idx=0, layerdrop=0.1):
        super().__init__()
        self.pool_weight_logit = nn.Parameter(torch.tensor(0.85))
        self.d_model = d_model
        self.use_cls = use_cls_token
        self.layerdrop = layerdrop
        self.pos_encoder=pos_encoder

        self.token_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.emb_ln = nn.LayerNorm(d_model)
        self.emb_dropout = nn.Dropout(dropout)

        if self.use_cls:
            self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
            nn.init.normal_(self.cls_token, mean=0.0, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.mlp = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )

    def forward(self, src, src_key_padding_mask):
        if src_key_padding_mask.dtype != torch.bool:
            src_key_padding_mask = src_key_padding_mask.bool()

        B, L = src.shape
        x = self.token_emb(src) * math.sqrt(self.d_model)
        x = x.transpose(0, 1)
        x = self.pos_encoder(x)
        x = self.emb_ln(x)
        x = self.emb_dropout(x)

        if self.use_cls:
            cls = self.cls_token.expand(1, B, -1).to(x.device)
            x = torch.cat([cls, x], dim=0)
            cls_mask = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
            src_key_padding_mask = torch.cat([cls_mask, src_key_padding_mask], dim=1)
  
        for layer in self.encoder.layers:
            
            if self.training and torch.rand(1).item() < self.layerdrop:
                continue
            x = layer(x, src_key_padding_mask=src_key_padding_mask)
    
        memory = self.encoder.norm(x) if self.encoder.norm is not None else x
        
        #memory = self.encoder(x, src_key_padding_mask=src_key_padding_mask)
        if self.use_cls:
            cls_rep  = memory[0]  # CLS token pooling
            # cls_rep + mean_rep
            mem = memory[1:].transpose(0, 1)  # [B, L, d_model] 
            mask = ~src_key_padding_mask[:, 1:mem.size(1)+1]
            denom = mask.sum(1, keepdim=True).clamp_min(1)
            mean_rep = (mem * mask.unsqueeze(-1)).sum(1) / denom
            
            weight = torch.sigmoid(self.pool_weight_logit)
            rep = weight * cls_rep + (1 - weight) * mean_rep
        else:
            mem = memory.transpose(0, 1)                     # [B, L, d_model]
            mask = ~src_key_padding_mask[:, :mem.size(1)]    # True=keep
            denom = mask.sum(1, keepdim=True).clamp_min(1)
            rep = (mem * mask.unsqueeze(-1)).sum(1) / denom  # masked mean pooling
        
        logits = self.mlp(rep).squeeze(-1)
        return logits

In [11]:
# ============ 7. Hyperpamaters search v2============
def objective(trial):
    # need train_folds, val_folds, tr_df, collate, g, vocab
    global evaluated_combinations
    
    gc.collect()
    torch.cuda.empty_cache()
    
    max_attempts = 50
    for attempt in range(max_attempts):
        nhead = trial.suggest_categorical("nhead", [2, 4, 8])
        d_model = trial.suggest_categorical("d_model", [64, 128, 256])
        dim_feedforward = trial.suggest_categorical("dim_feedforward", [256, 512, 1024])
        dropout = trial.suggest_categorical("dropout", [0.1, 0.2, 0.3, 0.4])
        num_layers = trial.suggest_categorical("num_layers", [2, 3, 4, 6])
        layerdrop = trial.suggest_categorical("layerdrop", [0.1, 0.2, 0.3])
        # ===== 3. 检查是否重复 =====
        params_tuple = (nhead, d_model, dim_feedforward, dropout, num_layers, layerdrop)
        if params_tuple not in evaluated_combinations:
            evaluated_combinations.add(params_tuple)
            break
    else:
        print(f" Trial {trial.number}: reached max attempts ({max_attempts}), skipping this trial.")
        raise optuna.TrialPruned()
    
    lr = 1e-4
    epochs = 30
    val_scores = []
    
    # ===== 4. K-fold =====
    n_splits = len(train_folds)
    for fold in range(n_splits):
        train_idx = train_folds[fold] 
        val_idx = val_folds[fold]

        train_sub_df = tr_df.iloc[train_idx].reset_index(drop=True)
        val_sub_df = tr_df.iloc[val_idx].reset_index(drop=True)

        # ===== 5. DataLoader =====
        train_dataset = SMILESDataset(train_sub_df)
        val_dataset = SMILESDataset(val_sub_df)
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate,generator=g)
        val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate)

        d_model = int(d_model)
        max_seq_len = 1000
        pos_encoder = PositionalEncoding(d_model=d_model, max_len=max_seq_len, learn_scale=True).to(device)
        model = TransformerClassifier(
            vocab_size=len(vocab),
            pos_encoder=pos_encoder,
            nhead=nhead,
            d_model=d_model,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            num_layers=num_layers,
            layerdrop=layerdrop
        ).to(device)

        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.BCEWithLogitsLoss()
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        
        for epoch in range(epochs):
            model.train()
            for padded, pad_mask, labels in train_loader:
                padded, pad_mask, labels = padded.to(device), pad_mask.to(device), labels.to(device)
                optimizer.zero_grad()
                logits = model(padded, pad_mask)
                loss = criterion(logits, labels)
                loss.backward()
                optimizer.step()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scheduler.step()

        model.eval() 
        val_labels, val_preds = [], []
        with torch.no_grad():
            for padded, pad_mask, labels in val_loader:
                padded, pad_mask, labels = padded.to(device), pad_mask.to(device), labels.to(device)
                logits = model(padded, pad_mask)
                prob = torch.sigmoid(logits)
                pred = (prob > 0.5).float()
                val_labels.extend(labels.cpu().numpy())
                val_preds.extend(pred.cpu().numpy())
        # MCC
        mcc = matthews_corrcoef(val_labels, val_preds)
        val_scores.append(mcc)
        
        del model, optimizer, scheduler
        gc.collect()
        torch.cuda.empty_cache()

    return np.mean(val_scores)

In [12]:
# ============ 8. Youden_Index  ============
def find_best_threshold(fpr, tpr, thresholds):
    J = tpr - fpr  # Youden's J statistic
    best_idx = np.argmax(J)  # Index of the maximum J value
    best_threshold = thresholds[best_idx]
    return best_threshold, fpr[best_idx], tpr[best_idx], J[best_idx]

In [13]:
# ============ 9. Evaluate model  ============
def evaluate_model(model,data_loader,device,d_model,dataset_name):
    '''
    Use these values to plot roc cruve
    '''
    all_metrics_history = {
        "all_loss": [], "all_auc": [], "all_acc": [], "all_mcc": [], "all_f1": [], "all_pre": [], "all_se":[], "all_sp":[], "all_Youden_Index":[], "all_Best_Threshold":[]
    }
    model.eval()
    all_labels, all_preds, all_probs = [], [], [] # [the number of all molecules in whole set]
    total_loss = 0
    total = 0
    with torch.no_grad():
        for padded, pad_mask, labels in data_loader: # [B,L] [B,L] [B]
            # move to GPU
            padded = padded.to(device)
            pad_mask = pad_mask.to(device)
            labels = labels.to(device)
            
             
            logits = model(padded,pad_mask) # logits:[B] labels;[B]
            loss = criterion(logits,labels) # 标量
            
            prob = torch.sigmoid(logits) # [B]
            pred = (prob > 0.5).float() # [B]

            all_labels.extend(labels.detach().cpu().numpy())
            all_probs.extend(prob.detach().cpu().numpy())
            all_preds.extend(pred.detach().cpu().numpy())
            
            total_loss += loss.item()
            total += 1
            
    avg_loss = total_loss / total
    
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    se = tp / (tp + fn)
    sp = tn / (tn + fp)

    fpr_all, tpr_all, thresholds_all = roc_curve(all_labels, all_probs)
    best_threshold_all, best_fpr_all, best_tpr_all, J_all = find_best_threshold(fpr_all, tpr_all, thresholds_all)
    
    all_metrics_history["all_loss"].append(float(avg_loss))
    all_metrics_history["all_auc"].append(float(roc_auc_score(all_labels, all_probs)))
    all_metrics_history["all_acc"].append(float(accuracy_score(all_labels, all_preds)))
    all_metrics_history["all_mcc"].append(float(matthews_corrcoef(all_labels, all_preds)))
    all_metrics_history["all_f1"].append(float(f1_score(all_labels, all_preds)))
    all_metrics_history["all_pre"].append(float(precision_score(all_labels, all_preds)))
    all_metrics_history["all_se"].append(float(se))
    all_metrics_history["all_sp"].append(float(sp))
    all_metrics_history["all_Youden_Index"].append(float(best_threshold_all))
    all_metrics_history["all_Best_Threshold"].append(float(J_all))
    
    # save values
    pred_results = pd.DataFrame([all_labels,all_probs,all_preds],
                                index=['y','probs','preds']).T
    # save metrics
    all_df = pd.DataFrame(all_metrics_history)

    pred_results.to_csv(fr"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\values_all_{dataset_name}_64.csv", index=False)
    all_df.to_csv(fr"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\metrics_all_{dataset_name}_64.csv", index=False)

In [14]:
# 1) comfirm GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.get_device_name())

NVIDIA GeForce RTX 5070


In [15]:
# 2) load data
molecules = pd.read_csv(r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\input\Valid_12756.csv")
molecules.shape

(12756, 5)

In [16]:
# 3) set global random seed
set_seed(64)  # 8,16,24,32,48,64

In [17]:
# 4) split data to train_df val_df and test_df (7：3)
train_val_df,te_df = train_test_split(molecules,test_size=0.1,random_state=64,stratify=molecules['Label'])
tr_df,val_df = train_test_split(train_val_df,test_size=0.1,random_state=64,stratify=train_val_df['Label'])
print(tr_df.shape)
print(val_df.shape)
print(te_df.shape)
tr_df.to_csv(r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\tr_df_64.csv", index=False)
val_df.to_csv(r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\val_64.csv", index=False)
te_df.to_csv(r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\te_df_64.csv", index=False)

(10332, 5)
(1148, 5)
(1276, 5)


In [18]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=64)
train_folds, val_folds = [], []
for tr_idx, val_idx in skf.split(tr_df, tr_df["Label"]):
    train_folds.append(tr_idx)
    val_folds.append(val_idx)

In [19]:
# 5) load vocab
with open(r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\vocab.json", "r", encoding="utf-8") as f:
    vocab = json.load(f)

In [20]:
len(vocab)

76

In [21]:
# 6) Prepare data processing tools
g = torch.Generator()
g.manual_seed(64) 
collate = make_collate_fn(vocab, use_bos_eos=True)
evaluated_combinations = set()

In [19]:
# 7) hyperparameters seach
study = optuna.create_study(direction="maximize") # we need max mcc
study.optimize(objective, n_trials=50)  # run 50 times

[I 2025-11-09 14:10:16,589] A new study created in memory with name: no-name-05d4b6ed-863d-4a77-96bb-7b226bbde4e9
[I 2025-11-09 14:16:47,417] Trial 0 finished with value: 0.2427106747361424 and parameters: {'nhead': 2, 'd_model': 64, 'dim_feedforward': 512, 'dropout': 0.2, 'num_layers': 2, 'layerdrop': 0.3}. Best is trial 0 with value: 0.2427106747361424.
[I 2025-11-09 14:30:39,125] Trial 1 finished with value: 0.26863597433921094 and parameters: {'nhead': 2, 'd_model': 64, 'dim_feedforward': 1024, 'dropout': 0.2, 'num_layers': 6, 'layerdrop': 0.2}. Best is trial 1 with value: 0.26863597433921094.
[I 2025-11-09 14:37:35,781] Trial 2 finished with value: 0.35740942844139184 and parameters: {'nhead': 4, 'd_model': 128, 'dim_feedforward': 1024, 'dropout': 0.1, 'num_layers': 2, 'layerdrop': 0.2}. Best is trial 2 with value: 0.35740942844139184.
[I 2025-11-09 14:44:57,620] Trial 3 finished with value: 0.2543398386492668 and parameters: {'nhead': 2, 'd_model': 64, 'dim_feedforward': 256, 'dr

⚠️ Trial 13: reached max attempts (50), skipping this trial.
⚠️ Trial 14: reached max attempts (50), skipping this trial.


[I 2025-11-09 16:30:52,265] Trial 15 finished with value: 0.31395594508963204 and parameters: {'nhead': 8, 'd_model': 256, 'dim_feedforward': 256, 'dropout': 0.3, 'num_layers': 3, 'layerdrop': 0.1}. Best is trial 11 with value: 0.4429844460239809.
[I 2025-11-09 16:45:01,889] Trial 16 finished with value: 0.23625173240179587 and parameters: {'nhead': 8, 'd_model': 128, 'dim_feedforward': 256, 'dropout': 0.4, 'num_layers': 6, 'layerdrop': 0.1}. Best is trial 11 with value: 0.4429844460239809.
[I 2025-11-09 16:55:34,325] Trial 17 finished with value: 0.4331203629705048 and parameters: {'nhead': 8, 'd_model': 256, 'dim_feedforward': 256, 'dropout': 0.1, 'num_layers': 4, 'layerdrop': 0.1}. Best is trial 11 with value: 0.4429844460239809.
[I 2025-11-09 17:04:38,923] Trial 18 finished with value: 0.38789227864067133 and parameters: {'nhead': 4, 'd_model': 256, 'dim_feedforward': 256, 'dropout': 0.1, 'num_layers': 4, 'layerdrop': 0.3}. Best is trial 11 with value: 0.4429844460239809.
[I 2025-1

⚠️ Trial 21: reached max attempts (50), skipping this trial.
⚠️ Trial 22: reached max attempts (50), skipping this trial.


[I 2025-11-09 17:25:41,044] Trial 23 pruned. 
[I 2025-11-09 17:25:41,169] Trial 24 pruned. 


...

⚠️ Trial 42: reached max attempts (50), skipping this trial.
⚠️ Trial 43: reached max attempts (50), skipping this trial.


[I 2025-11-09 19:04:26,546] Trial 44 finished with value: 0.39272061062737734 and parameters: {'nhead': 2, 'd_model': 256, 'dim_feedforward': 256, 'dropout': 0.1, 'num_layers': 2, 'layerdrop': 0.1}. Best is trial 11 with value: 0.4429844460239809.
[I 2025-11-09 19:19:15,111] Trial 45 finished with value: 0.2785840676445687 and parameters: {'nhead': 8, 'd_model': 256, 'dim_feedforward': 256, 'dropout': 0.4, 'num_layers': 6, 'layerdrop': 0.1}. Best is trial 11 with value: 0.4429844460239809.
[I 2025-11-09 19:19:15,210] Trial 46 pruned. 


⚠️ Trial 46: reached max attempts (50), skipping this trial.


[I 2025-11-09 19:25:35,239] Trial 47 finished with value: 0.36268878679653 and parameters: {'nhead': 8, 'd_model': 256, 'dim_feedforward': 1024, 'dropout': 0.1, 'num_layers': 2, 'layerdrop': 0.3}. Best is trial 11 with value: 0.4429844460239809.
[I 2025-11-09 19:38:35,217] Trial 48 finished with value: 0.30149428006860385 and parameters: {'nhead': 4, 'd_model': 128, 'dim_feedforward': 256, 'dropout': 0.2, 'num_layers': 6, 'layerdrop': 0.2}. Best is trial 11 with value: 0.4429844460239809.
[I 2025-11-09 19:38:35,311] Trial 49 pruned. 


⚠️ Trial 49: reached max attempts (50), skipping this trial.


In [20]:
# 8) get best hyperparameters
results = []
for t in study.trials:
    trial_result = t.params.copy()   # 
    trial_result["value"] = t.value  # loss
    results.append(trial_result)

params_df = pd.DataFrame(results)
params_df.to_csv(r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\all_params_results_64.csv", index=False)

best_params_df = pd.DataFrame([study.best_params])
best_params_df.to_csv(r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\best_params_result_64.csv", index=False)

In [22]:
# 9) define the Dataset
train_dataset = SMILESDataset(tr_df)
val_dataset = SMILESDataset(val_df)
test_dataset = SMILESDataset(te_df)

In [23]:
# 10) define the Dataloader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate,generator=g)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True, collate_fn=collate,generator=g)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate,generator=g)

In [ ]:
study.best_params

In [25]:
# 11) Use best hyperparameters to define model
nhead=study.best_params['nhead']
d_model = study.best_params['d_model']
dim_feedforward=study.best_params['dim_feedforward']
dropout=study.best_params['dropout']
layerdrop=study.best_params['layerdrop']
num_layers=study.best_params['num_layers']

max_seq_len = 1000
pos_encoder = PositionalEncoding(d_model=d_model, max_len=max_seq_len, learn_scale=True).to(device)
model = TransformerClassifier(vocab_size=len(vocab), 
                              pos_encoder=pos_encoder,
                              nhead=nhead,
                              d_model=d_model,
                              dim_feedforward=dim_feedforward,
                              dropout=dropout,
                              num_layers=num_layers,
                              layerdrop=layerdrop).to(device)

In [26]:
# 12) set the optimizer and criterion
epochs = 100
lr = 0.0001
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss() # label must be float type
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [34]:
# 13) train the model
train_metrics_history = {
    "tr_loss": [], "tr_auc": [], "tr_acc": [], "tr_mcc": [], "tr_f1": [], "tr_pre": [], "tr_se":[], "tr_sp":[], "tr_Youden_Index":[], "tr_Best_Threshold":[]
}
val_metrics_history = {
    "val_loss": [], "val_auc": [], "val_acc": [], "val_mcc": [], "val_f1": [], "val_pre": [], "val_se":[], "val_sp":[], "val_Youden_Index":[], "val_Best_Threshold":[]
}

best_val_mcc = 0.0
patience = 10                
counter = 0                  
best_model_state = None       

for epoch in range(epochs):
    model.train()
    tr_labels, tr_preds, tr_probs = [], [], []
    train_total_loss = 0
    train_total = 0

    for padded, pad_mask, labels in train_loader:
        
        # move to GPU
        padded = padded.to(device)
        pad_mask = pad_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(padded,pad_mask) # logits:[B] labels;[B]
        loss = criterion(logits,labels)
        loss.backward()
        optimizer.step()
        
        prob = torch.sigmoid(logits) # [B]
        pred = (prob > 0.5).float() # [B]
        
        tr_labels.extend(labels.detach().cpu().numpy())
        tr_probs.extend(prob.detach().cpu().numpy())
        tr_preds.extend(pred.detach().cpu().numpy())
        
        train_total_loss += loss.item()
        train_total += 1
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
    scheduler.step()   
    
    train_avg_loss = train_total_loss / train_total
    tn, fp, fn, tp = confusion_matrix(tr_labels, tr_preds).ravel()
    se = tp / (tp + fn)
    sp = tn / (tn + fp)

    fpr_train, tpr_train, thresholds_train = roc_curve(tr_labels, tr_probs)
    best_threshold_train, best_fpr_train, best_tpr_train, J_train = find_best_threshold(fpr_train, tpr_train, thresholds_train)
    
    train_metrics_history["tr_loss"].append(float(train_avg_loss))
    train_metrics_history["tr_auc"].append(float(roc_auc_score(tr_labels, tr_probs)))
    train_metrics_history["tr_acc"].append(float(accuracy_score(tr_labels, tr_preds)))
    train_metrics_history["tr_mcc"].append(float(matthews_corrcoef(tr_labels, tr_preds)))
    train_metrics_history["tr_f1"].append(float(f1_score(tr_labels, tr_preds)))
    train_metrics_history["tr_pre"].append(float(precision_score(tr_labels, tr_preds)))
    train_metrics_history["tr_se"].append(float(se))
    train_metrics_history["tr_sp"].append(float(sp))
    train_metrics_history["tr_Youden_Index"].append(float(J_train))
    train_metrics_history["tr_Best_Threshold"].append(float(best_threshold_train))
    
    model.eval()
    val_labels, val_preds, val_probs = [], [], [] # [the number of all molecules in validation set]
    val_total_loss = 0
    val_total = 0
    with torch.no_grad():
        for padded, pad_mask, labels in val_loader: # [B,L] [B,L] [B]
            # move to GPU
            padded = padded.to(device)
            pad_mask = pad_mask.to(device)
            labels = labels.to(device)
            
            #pos_encoder = PositionalEncoding(d_model, padded.shape[1]+1).to(device)
            logits = model(padded,pad_mask) # logits:[B] labels;[B]
            loss = criterion(logits,labels)
            
            prob = torch.sigmoid(logits) # [B]
            pred = (prob > 0.5).float() # [B]
            
            val_labels.extend(labels.detach().cpu().numpy())
            val_probs.extend(prob.detach().cpu().numpy())
            val_preds.extend(pred.detach().cpu().numpy())
            
            val_total_loss += loss.item()
            val_total += 1
            
    val_avg_loss = val_total_loss / val_total
    tn, fp, fn, tp = confusion_matrix(val_labels, val_preds).ravel()
    se = tp / (tp + fn)
    sp = tn / (tn + fp)

    fpr_val, tpr_val, thresholds_val = roc_curve(val_labels, val_probs)
    best_threshold_val, best_fpr_val, best_tpr_val, J_val = find_best_threshold(fpr_val, tpr_val, thresholds_val)
        
    val_metrics_history["val_loss"].append(float(val_avg_loss))
    val_metrics_history["val_auc"].append(float(roc_auc_score(val_labels, val_probs)))
    val_metrics_history["val_acc"].append(float(accuracy_score(val_labels, val_preds)))
    val_metrics_history["val_mcc"].append(float(matthews_corrcoef(val_labels, val_preds)))
    val_metrics_history["val_f1"].append(float(f1_score(val_labels, val_preds)))
    val_metrics_history["val_pre"].append(float(precision_score(val_labels, val_preds)))
    val_metrics_history["val_se"].append(float(se))
    val_metrics_history["val_sp"].append(float(sp))
    val_metrics_history["val_Youden_Index"].append(float(J_val))
    val_metrics_history["val_Best_Threshold"].append(float(best_threshold_val))
    #val_auc = roc_auc_score(val_labels, val_probs)
    val_mcc = matthews_corrcoef(val_labels, val_preds)

    if val_mcc > best_val_mcc:
        best_val_mcc = val_mcc
        best_model_state  = copy.deepcopy(model.state_dict())
        counter = 0
        print(f"Epoch {epoch} - Train Loss: {train_avg_loss:.4f} | Val Loss: {val_avg_loss:.4f} | Val MCC: {val_mcc:.4f}")
    else:
        counter += 1
        print(f"Epoch {epoch}: val_mcc did not improve for {counter} epochs.")
        print(f"Epoch {epoch} - Train Loss: {train_avg_loss:.4f} | Val Loss: {val_avg_loss:.4f} | Val MCC: {val_mcc:.4f}")
        if counter >= patience:
            print("Early stopping triggered.")
            break

if best_model_state is not None:
    model.load_state_dict(best_model_state)
    torch.save(model.state_dict(), r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\transformer_classifier_64_best.pth")

tr_df = pd.DataFrame(train_metrics_history)
val_df = pd.DataFrame(val_metrics_history)
tr_df.to_csv(r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\performance_tr_64.csv", index=False)
val_df.to_csv(r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\performance_val_64.csv", index=False)

Epoch 0 - Train Loss: 0.6719 | Val Loss: 0.6713 | Val MCC: 0.1679
Epoch 1 - Train Loss: 0.6565 | Val Loss: 0.6623 | Val MCC: 0.1909
Epoch 2 - Train Loss: 0.6393 | Val Loss: 0.6472 | Val MCC: 0.2480
Epoch 3: val_mcc did not improve for 1 epochs.
Epoch 3 - Train Loss: 0.6300 | Val Loss: 0.6362 | Val MCC: 0.2411
Epoch 4 - Train Loss: 0.6252 | Val Loss: 0.6295 | Val MCC: 0.2833
Epoch 5 - Train Loss: 0.6159 | Val Loss: 0.6248 | Val MCC: 0.2935
Epoch 6: val_mcc did not improve for 1 epochs.
Epoch 6 - Train Loss: 0.6086 | Val Loss: 0.6194 | Val MCC: 0.2902
Epoch 7 - Train Loss: 0.6003 | Val Loss: 0.6235 | Val MCC: 0.3084
Epoch 8 - Train Loss: 0.5954 | Val Loss: 0.6151 | Val MCC: 0.3380
Epoch 9 - Train Loss: 0.5853 | Val Loss: 0.6017 | Val MCC: 0.3815
Epoch 10: val_mcc did not improve for 1 epochs.
Epoch 10 - Train Loss: 0.5837 | Val Loss: 0.6074 | Val MCC: 0.3533
Epoch 11: val_mcc did not improve for 2 epochs.
Epoch 11 - Train Loss: 0.5698 | Val Loss: 0.6122 | Val MCC: 0.3352
Epoch 12 - Train

In [35]:
train_results = evaluate_model(model, train_loader, device, d_model, "train_set")
test_results = evaluate_model(model, test_loader, device, d_model, "test_set")

In [27]:
model.load_state_dict(torch.load(
    r"E:\Projects\Mycobacterium tuberculosis\03-Machine Learning\result\64\Hyperparameters Rank 3\transformer_classifier_64_best.pth",
    map_location=device  
))

<All keys matched successfully>